In [47]:
import sys
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt
from importlib import reload

In [2]:
sys.path.append("../../src")
import main
import vis

In [ ]:
reload(vis)

In [4]:
temp = pd.read_csv('../../data/raw_data/ta_vp_rh_60_min_2022_09_01_2024_08_31_gap_filled_new.csv')
temp['datetime']=pd.to_datetime(temp['datetime'])
temp['datetime_UTC'] = temp['datetime']
temp['value'] = temp['ta']
temp = temp[temp['type'] == 'measured']

In [5]:
temp = temp[temp['station_id'] != 'FREICH']
temp = temp[temp['station_id'] != 'FRWITT']

In [ ]:
temp

In [7]:
stations = pd.read_csv("../../data/raw_data/Freiburg-Street-Level-Weather-Station-Network-MetaData-V1-0.csv")

In [8]:
temp['month'] = temp['datetime_UTC'].dt.month

In [ ]:
from suncalc import get_position, get_times
from datetime import datetime, timezone
lon = 7.85222
lat = 47.9959

times = {}

temp['Time'] = pd.to_datetime(temp['datetime'])

for i in temp['Time'].unique():

    times[i] = get_times(i, lon, lat)

temp['sunrise'] = temp['Time'].apply(
    lambda x: times[x]['sunrise'].replace(tzinfo=timezone.utc))
temp['sunset'] = temp['Time'].apply(
    lambda x: times[x]['sunset'].replace(tzinfo=timezone.utc))

temp['time_of_day'] = np.where(
    temp['Time'] < temp['sunrise'], 'night',
    np.where(temp['Time'] > temp['sunset'], 'night', 'day'))

temp_n = temp[temp['time_of_day'].isin(['night'])]

# Temperature Minimum and Maximum

In [ ]:
temp_n.idxmin()

In [ ]:
temp_n.loc[109121]

In [76]:
jan = temp_n[temp_n['month'] == 1]
dec = temp_n[temp_n['month'] == 12]
feb = temp_n[temp_n['month'] == 2]

In [ ]:
jan['value'].mean(), dec['value'].mean(), feb['value'].mean()

In [ ]:
jul = temp_n[temp_n['month'] == 7]
aug = temp_n[temp_n['month'] == 8]
sep = temp_n[temp_n['month'] == 9]

jul['value'].mean(), aug['value'].mean(), sep['value'].mean()

In [79]:
mean = temp_n['value'].mean()

In [ ]:
mean

In [81]:
std = temp_n['value'].std()

In [ ]:
std

# Diurnal Amplitude

In [18]:
# for each day, do max - min
temp['day'] = temp['datetime_UTC'].dt.date
temp['day'] = pd.to_datetime(temp['day'])
daily = temp.groupby(['day', 'station_id']).agg({'value': ['max', 'min']})
daily.columns = ['max', 'min']
daily = daily.reset_index()
daily['range'] = daily['max'] - daily['min']

In [19]:
# drop 2024-09-01
daily = daily[daily['day'] != '2024-09-01']

In [ ]:
daily['range'].mean()

In [ ]:
daily[daily['station_id'] == 'FRKART']['range'].mean()


In [ ]:
daily[daily['station_id'] == 'FRHOCH']['range'].mean()

# Temperature Plots

In [ ]:
temp_n

In [ ]:
temp_n

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax2 = ax.twinx()
# Histogram
binwidth =0.5
data = temp['value']
ax.hist(temp['value'], bins=np.arange(min(data), max(data) + binwidth, binwidth), density=False, alpha=0.6, color='steelblue', edgecolor='black')
ax2.hist(temp['value'], bins=np.arange(min(data), max(data) + binwidth, binwidth), density=True, alpha=0.0, color='steelblue', edgecolor='black')

# Overlay normal distribution
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 500)
p = norm.pdf(x, mean, std)
ax2.plot(x, p, 'r', linewidth=2, label='Normal')

#put N in plot
n = len(temp['value'])
ax.text(0.76, 0.87, f'N = {n}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Number of Hourly Measurements\n(station-hours)")
ax2.set_ylabel("Probability Density")
plt.legend()
plt.grid()
plt.savefig('../../figures/fig4/temp_dist_nov.png',bbox_inches='tight', dpi=300)
plt.savefig('../../figures/fig4/temp_dist_nov.svg',bbox_inches='tight')
plt.savefig('../../figures/fig4/temp_dist_nov.pdf',bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax2 = ax.twinx()
# Histogram
binwidth =0.5
data = temp_n['value']
ax.hist(temp_n['value'], bins=np.arange(min(data), max(data) + binwidth, binwidth), density=False, alpha=0.6, color='steelblue', edgecolor='black')
ax2.hist(temp_n['value'], bins=np.arange(min(data), max(data) + binwidth, binwidth), density=True, alpha=0.0, color='steelblue', edgecolor='black')

# add mean and median lines
ax.axvline(mean, color='red', linewidth=2, label='Mean')

#put N in plot
n = len(temp_n['value'])
ax.text(0.76, 0.87, f'N = {n}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Number of Hourly Measurements\n(station-hours)")
ax2.set_ylabel("Probability Density")
ax.legend()
plt.grid()
plt.savefig('../../figures/fig4/temp_dist_nov.png',bbox_inches='tight', dpi=300)
plt.savefig('../../figures/fig4/temp_dist_nov.svg',bbox_inches='tight')
plt.savefig('../../figures/fig4/temp_dist_nov.pdf',bbox_inches='tight')
plt.show()

In [ ]:
stats.normaltest(temp['value'])

In [ ]:
temp['value'].skew()

# CL-UHI Magnitude

In [51]:
var = 'BuAre_sum'

In [52]:
stats_dict = pd.read_csv(f'../../data/processed_data/calplot_data/{var}_300_fraction_of_night.csv')

In [53]:
stats = pd.read_csv(f'../../data/processed_data/2024/stats_timesteps_{var}_year_300.csv')

In [ ]:
stats_dict

In [55]:
uhi_n = stats_dict[(stats_dict['fraction_of_night'] < 1) & (stats_dict['fraction_of_night'] > 0)]['UHI Magnitude']

In [57]:
mean_uhi = uhi_n.mean()
std_uhi = uhi_n.std()

In [ ]:
mean_uhi, std_uhi

In [ ]:
np.median(uhi_n.dropna())

In [ ]:
uhi_n.dropna().mode()

In [ ]:
mode_uhi

In [ ]:
uhi_n

In [65]:
# redo calplot

In [ ]:
mean_uhi, median_uhi

In [ ]:
uhi_n

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax2 = ax.twinx()
data= uhi_n
binwidth = 0.1
ax.hist(uhi_n, bins=np.arange(min(data), max(data) + binwidth, binwidth), density=False, alpha=0.6, color='steelblue', edgecolor='black')
ax2.hist(uhi_n, bins=np.arange(min(data), max(data) + binwidth, binwidth), density=True, alpha=0.6, color='steelblue', edgecolor='black')
# Histogram

# Overlay mean and median
ax.axvline(mean_uhi, color='red', linewidth=2, label='Mean')
median_uhi = np.median(uhi_n.dropna())
ax.axvline(median_uhi, color='blue', linewidth=2, label='Median')

n = len(uhi_n)
ax.text(0.76, 0.87, f'N = {n}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

ax.set_xlabel("Nocturnal CL-UHI Magnitude (K)")
ax.set_ylabel("Number of Hourly Measurements\n(station-hours)")
ax2.set_ylabel("Probability Density")

ax.legend(loc='upper right', ncol=2)
plt.xlim(-3)
plt.grid(True)
plt.savefig('../../figures/fig4/uhi_dist_new.png',bbox_inches='tight')
plt.savefig('../../figures/fig4/uhi_dist_new.svg',bbox_inches='tight')
plt.savefig('../../figures/fig4/uhi_dist_new.pdf',bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax2 = ax.twinx()
data= uhi_n
binwidth = 0.1
ax.hist(uhi_n, bins=np.arange(min(data), max(data) + binwidth, binwidth), density=False, alpha=0.6, color='steelblue', edgecolor='black')
ax2.hist(uhi_n, bins=np.arange(min(data), max(data) + binwidth, binwidth), density=True, alpha=0.6, color='steelblue', edgecolor='black')
# Histogram

# Overlay normal distribution
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 500)
p = norm.pdf(x, mean_uhi, std_uhi)
plt.plot(x, p, 'r', linewidth=2, label='Normal')

n = len(uhi_n)
ax.text(0.76, 0.87, f'N = {n}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

ax.set_xlabel("Nocturnal CL-UHI Magnitude (K)")
ax.set_ylabel("Number of Hourly Measurements\n(station-hours)")
ax2.set_ylabel("Probability Density")

plt.legend()
plt.grid(True)
plt.savefig('../../figures/fig4/uhi_dist_new.png',bbox_inches='tight')
plt.savefig('../../figures/fig4/uhi_dist_new.svg',bbox_inches='tight')
plt.savefig('../../figures/fig4/uhi_dist_new.pdf',bbox_inches='tight')
plt.show()

In [ ]:
uhi_n

Summer/Winter

In [ ]:
stats_dict

In [106]:
stats_dict['Time'] = pd.to_datetime(stats_dict['Time'])

In [108]:
stats_dict['month'] = stats_dict['Time'].dt.month

In [ ]:
stats_dict[(stats_dict['fraction_of_night'] < 1) & (stats_dict['fraction_of_night'] > 0)]['UHI Magnitude'].plot.hist(bins=100, density=True, alpha=0.6, color='steelblue', edgecolor='black')

In [ ]:
stats_dict[(stats_dict['fraction_of_night'] < 1) & (stats_dict['fraction_of_night'] > 0)][stats_dict['month'].isin([6, 7, 8])]['UHI Magnitude'].mean()

In [ ]:
stats_dict[(stats_dict['fraction_of_night'] < 1) & (stats_dict['fraction_of_night'] > 0)][stats_dict['month'].isin([12, 1, 2])]['UHI Magnitude'].mean()